# Skater → 3D model (Hunyuan3D-2mv on Kaggle)

Turns 4 screenshots of the character (front / back / left / right) into a textured `.glb`.

**Before running**
1. Kaggle → *Create → New Notebook* → *File → Import Notebook* → pick this file.
2. Right sidebar → *Session options*: **Accelerator = GPU T4 x2**, **Internet = On** (needs a phone-verified Kaggle account).
3. Right sidebar → *Add Input → Upload* → new dataset containing your images named
   `front.png`, `back.png`, `left.png`, `right.png` (jpg/webp fine too; `back`/`right` optional).
   - Pose the character in a **T-pose or A-pose** (arms away from body, small gap between legs) — otherwise rigging in Mixamo breaks.
   - Same zoom, lighting and plain background in every shot. Backgrounds are removed automatically.
4. *Run All*. Takes ~15–25 min the first time (install + model download).

**Outputs** (right sidebar → *Output*, or `/kaggle/working`):
- `skater_shape.glb` — untextured mesh (always produced, backup)
- `skater_textured.glb` — final model → put this in the game's `public/` folder

In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Get Hunyuan3D-2 and install it (kept in /tmp so it doesn't bloat the Output)
REPO = '/tmp/Hunyuan3D-2'
!git clone --depth 1 https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git {REPO}
!pip install -q -r {REPO}/requirements.txt
!pip install -q -e {REPO}

In [ ]:
# 3. Build the texture-painting CUDA extensions (needed only for the textured model; ~5 min)
!cd {REPO}/hy3dgen/texgen/custom_rasterizer && python setup.py install > /tmp/rast.log 2>&1 && echo 'custom_rasterizer OK' || tail -30 /tmp/rast.log
!cd {REPO}/hy3dgen/texgen/differentiable_renderer && python setup.py install > /tmp/dr.log 2>&1 && echo 'differentiable_renderer OK' || tail -30 /tmp/dr.log

In [ ]:
# 4. Find and prep the images
import sys, glob, os
sys.path.insert(0, REPO)
from PIL import Image
from hy3dgen.rembg import BackgroundRemover

# If the model comes out mirrored/scrambled, swap these two and re-run from this cell.
SWAP_LEFT_RIGHT = False

found = {}
for path in glob.glob('/kaggle/input/**/*', recursive=True):
    name, ext = os.path.splitext(os.path.basename(path).lower())
    if ext in ('.png', '.jpg', '.jpeg', '.webp') and name in ('front', 'back', 'left', 'right'):
        found[name] = path
if SWAP_LEFT_RIGHT:
    found['left'], found['right'] = found.get('right'), found.get('left')
    found = {k: v for k, v in found.items() if v}
assert 'front' in found, f'Need at least a front image. Found: {found}'
print('Using:', found)

rembg = BackgroundRemover()
images = {}
for view, path in found.items():
    img = Image.open(path)
    has_alpha = img.mode == 'RGBA' and img.getchannel('A').getextrema()[0] < 255
    img = img.convert('RGBA') if has_alpha else rembg(img.convert('RGB'))
    images[view] = img
    img.save(f'/kaggle/working/cutout_{view}.png')  # check these look clean
print('Background removed; previews saved as cutout_*.png')

In [ ]:
# 5. Generate the shape from all views
import torch
from hy3dgen.shapegen import (Hunyuan3DDiTFlowMatchingPipeline, FloaterRemover,
                              DegenerateFaceRemover, FaceReducer)

shape = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    'tencent/Hunyuan3D-2mv', subfolder='hunyuan3d-dit-v2-mv', variant='fp16')

mesh = shape(
    image=images,
    num_inference_steps=50,
    octree_resolution=380,
    num_chunks=20000,
    generator=torch.manual_seed(12345),
    output_type='trimesh',
)[0]

mesh = FloaterRemover()(mesh)
mesh = DegenerateFaceRemover()(mesh)
mesh = FaceReducer()(mesh)  # ~40k faces: light enough for the game and for Mixamo
mesh.export('/kaggle/working/skater_shape.glb')
print('Saved skater_shape.glb —', len(mesh.faces), 'faces')

del shape
torch.cuda.empty_cache()

In [ ]:
# 6. Paint textures onto it (uses the front image as the colour reference)
from hy3dgen.texgen import Hunyuan3DPaintPipeline

paint = Hunyuan3DPaintPipeline.from_pretrained('tencent/Hunyuan3D-2')
try:
    paint.enable_model_cpu_offload()  # keeps it inside the T4's 15 GB
except Exception as e:
    print('cpu offload not available:', e)

textured = paint(mesh, image=images['front'])
textured.export('/kaggle/working/skater_textured.glb')
print('Saved skater_textured.glb')

## Next
1. Download `skater_textured.glb` from the Output panel.
2. Rig it at **mixamo.com** (if it rejects the GLB, import into Blender → export FBX). Download **without animation**, FBX or convert back to GLB in Blender.
3. Drop it in the game's `public/` folder and ask Claude to wire it up.

**Troubleshooting**
- Out of memory in step 6 → *Run → Restart*, then re-run cells 2 (instant, already installed), 4, 5 and 6 — nothing else is on the GPU then.
- Step 3 failed → you still get `skater_shape.glb`; colour can be painted in Blender, or retry on a fresh session.
- Weird/mirrored shape → set `SWAP_LEFT_RIGHT = True` in cell 4 and re-run 4–6.